# Lección 5: Machine Learning Escalable con Spark MLlib
## Pipeline Completo: Clasificación + Segmentación para RetailMax

---

### Módulo 9 — Retail Analytics Pipeline | Proyecto Final

---

## 📋 Recap del proyecto: Lecciones 1–4

| Lección | Tema | Herramienta principal | Resultado |
|---------|------|----------------------|----------|
| **1** | Big Data en el e-commerce | Teoría + Arquitectura | Comprensión del problema de escala en RetailMax |
| **2** | Apache Spark — Introducción | SparkSession, RDDs básicos | Primer job distribuido |
| **3** | RDDs en profundidad | transformaciones, acciones | Pipeline de conteo y agregación de productos |
| **4** | DataFrames, SQL y Parquet | Spark SQL, `.parquet()` | `leccion4_fashion_data.parquet` + `leccion4_metricas.parquet` |
| **5 (esta)** | Machine Learning con MLlib | Pipeline, LR, K-Means | Clasificador + Segmentador de catálogo RetailMax |

---

## 🎯 Objetivo de esta lección

RetailMax necesita **automatizar** dos tareas en su catálogo de moda de 70,000 imágenes:

1. **Clasificación automática** → ¿Es este producto "ropa de cuerpo" (camisetas, vestidos, abrigos) o "accesorio/calzado" (bolsos, zapatillas, sandalias)? Usaremos **Regresión Logística**.

2. **Segmentación de productos** → ¿En qué grupos visuales se agrupan los productos del catálogo? Usaremos **K-Means** para identificar segmentos de mercado.

Ambos modelos se construirán usando el patrón **MLlib Pipeline**, que encadena pasos de preprocesamiento y modelado de forma reproducible y escalable.

---

## 📦 Dataset: Fashion-MNIST

- **70,000 imágenes** de 28×28 píxeles (grayscale)
- **10 categorías**: T-shirt/Top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
- **Features extraídas** (de la Lección 4): `pixel_mean`, `pixel_std`, `pixel_max`, `pixel_min`

> **Nota de diseño**: En lugar de usar los 784 píxeles como features (lo cual sería costoso), usamos 4 estadísticas agregadas de cada imagen. Esto es un truco muy común en visión computacional industrial: reducir dimensionalidad manteniendo información relevante.

---
# Sección 0: Configuración Inicial y Carga de Datos

Importamos todas las librerías necesarias, creamos la SparkSession e intentamos cargar los datos de la Lección 4. Si el archivo Parquet no existe, reconstruimos los datos automáticamente.

In [1]:
# ============================================================
# SECCIÓN 0: Configuración inicial e importaciones
# ============================================================

# --- Librerías estándar ---
import os           # manejo de rutas y directorios
import time         # medir tiempos de entrenamiento
import warnings     # suprimir advertencias menores
warnings.filterwarnings('ignore')

# --- Librerías de datos y visualización ---
import numpy as np          # cálculos numéricos
import pandas as pd         # DataFrames locales (para visualizaciones)
import matplotlib.pyplot as plt   # gráficas
import matplotlib.patches as mpatches
import seaborn as sns       # heatmaps y gráficas estadísticas

# Ajustar estilos de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# --- Spark Core ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F      # funciones SQL (when, col, isin, etc.)
from pyspark.sql.types import (             # tipos de datos Spark
    StructType, StructField,
    IntegerType, FloatType, StringType
)

# --- MLlib: Feature Engineering ---
from pyspark.ml import Pipeline            # orquesta pasos en secuencia
from pyspark.ml.feature import (
    VectorAssembler,   # combina columnas en un vector
    StandardScaler,    # estandariza features (z-score)
    StringIndexer      # convierte strings a índices numéricos
)

# --- MLlib: Modelos ---
from pyspark.ml.classification import LogisticRegression   # clasificación binaria
from pyspark.ml.clustering import KMeans                   # segmentación no supervisada

# --- MLlib: Evaluación ---
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,  # accuracy, F1, precision, recall
    ClusteringEvaluator                 # silhouette score
)

# --- MLlib: Optimización de hiperparámetros ---
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

print("✅ Importaciones completadas exitosamente")

✅ Importaciones completadas exitosamente


In [2]:
# ============================================================
# PASO 2: Crear SparkSession
# ============================================================
# La SparkSession es el punto de entrada a todo Spark.
# getOrCreate() reutiliza la sesión existente si ya hay una activa.

spark = (
    SparkSession.builder
    .appName("RetailMax_MLlib_Leccion5")    # nombre del job en la UI
    .config("spark.sql.shuffle.partitions", "8")  # reducir particiones para datasets medianos
    .config("spark.driver.memory", "2g")    # memoria para el driver
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")  # compatibilidad Windows
    .getOrCreate()
)

# Reducir verbosidad de los logs de Spark
spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession activa")
print(f"   Versión de Spark : {spark.version}")
print(f"   App Name         : {spark.sparkContext.appName}")
print(f"   Master           : {spark.sparkContext.master}")

✅ SparkSession activa
   Versión de Spark : 4.1.1
   App Name         : RetailMax_MLlib_Leccion5
   Master           : local[*]


In [3]:
# ============================================================
# PASO 3: Cargar datos desde Parquet (Lección 4)
#         Si no existe, reconstruir desde Fashion-MNIST
# ============================================================

PARQUET_PATH = "output/leccion4_fashion_data.parquet"  # ruta del archivo de la Lección 4

# Función auxiliar para generar datos sintéticos con la misma estructura
def generar_datos_sinteticos(spark_session, n=70000):
    """
    Genera datos sintéticos que imitan Fashion-MNIST con las
    estadísticas de píxeles que construimos en la Lección 4.
    """
    print("  ⚙️  Generando datos sintéticos (réplica de Fashion-MNIST)...")
    np.random.seed(42)  # reproducibilidad

    # Mapeo de etiquetas a nombres (igual que Fashion-MNIST original)
    label_names = {
        0: "T-shirt/Top",
        1: "Trouser",
        2: "Pullover",
        3: "Dress",
        4: "Coat",
        5: "Sandal",
        6: "Shirt",
        7: "Sneaker",
        8: "Bag",
        9: "Ankle Boot"
    }

    # Parámetros por clase (media y std de pixel_mean para cada categoría)
    # Basados en estadísticas reales del dataset Fashion-MNIST
    class_params = {
        0: (72.9, 18.0),   # T-shirt/Top   — gris medio
        1: (80.1, 16.0),   # Trouser       — gris medio-claro
        2: (74.6, 19.0),   # Pullover      — gris medio
        3: (68.4, 22.0),   # Dress         — más oscuro, mayor varianza
        4: (71.2, 20.0),   # Coat          — gris oscuro
        5: (60.5, 25.0),   # Sandal        — fondo claro, objeto oscuro
        6: (76.3, 18.5),   # Shirt         — similar a T-shirt
        7: (62.8, 24.0),   # Sneaker       — similar a sandal
        8: (55.1, 28.0),   # Bag           — mayor varianza
        9: (66.4, 23.0),   # Ankle Boot    — oscuro
    }

    rows = []
    n_per_class = n // 10  # 7000 imágenes por clase

    for label, (mu, sigma) in class_params.items():
        for i in range(n_per_class):
            # Simular pixel_mean con distribución normal truncada [0, 255]
            pmean = float(np.clip(np.random.normal(mu, sigma), 0, 255))
            # pixel_std: correlacionada con la clase
            pstd  = float(np.clip(np.random.normal(sigma * 2.5, sigma * 0.5), 1, 120))
            # pixel_max: siempre >= pixel_mean
            pmax  = float(np.clip(pmean + np.abs(np.random.normal(80, 20)), pmean, 255))
            # pixel_min: siempre <= pixel_mean
            pmin  = float(np.clip(pmean - np.abs(np.random.normal(60, 15)), 0, pmean))
            # split: 60k train, 10k test (como Fashion-MNIST original)
            split = "train" if i < (n_per_class * 6 // 7) else "test"

            image_id = label * n_per_class + i
            rows.append((image_id, label, label_names[label], pmean, pstd, pmax, pmin, split))

    # Crear schema explícito para mayor control
    schema = StructType([
        StructField("image_id",   IntegerType(), False),
        StructField("label",      IntegerType(), False),
        StructField("label_name", StringType(),  False),
        StructField("pixel_mean", FloatType(),   False),
        StructField("pixel_std",  FloatType(),   False),
        StructField("pixel_max",  FloatType(),   False),
        StructField("pixel_min",  FloatType(),   False),
        StructField("split",      StringType(),  False),
    ])

    # Crear DataFrame Spark desde lista de Python
    df = spark_session.createDataFrame(rows, schema=schema)
    return df


# ============================================================
# Estrategia de carga:
#   1) Intenta leer Parquet de Lección 4
#   2) Si falla, intenta cargar Fashion-MNIST desde tensorflow
#   3) Si falla, intenta pytorch / torchvision
#   4) Si falla, intenta sklearn fetch_openml
#   5) Último recurso: datos sintéticos
# ============================================================

df = None  # inicializar
fuente_datos = None

# --- Intento 1: Parquet de Lección 4 ---
print("Paso 1: Buscando 'output/leccion4_fashion_data.parquet'...")
try:
    if os.path.exists(PARQUET_PATH):
        df = spark.read.parquet(PARQUET_PATH)
        n_rows = df.count()
        print(f"  ✅ Parquet cargado. Filas: {n_rows:,}")
        fuente_datos = "Parquet Lección 4"
    else:
        print(f"  ⚠️  Archivo no encontrado en '{PARQUET_PATH}'")
except Exception as e:
    print(f"  ❌ Error leyendo Parquet: {e}")

# --- Intento 2: TensorFlow / Keras ---
if df is None:
    print("Paso 2: Intentando cargar desde TensorFlow/Keras...")
    try:
        import tensorflow as tf
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

        label_names_list = ["T-shirt/Top","Trouser","Pullover","Dress","Coat",
                            "Sandal","Shirt","Sneaker","Bag","Ankle Boot"]

        def imgs_to_rows(images, labels, split_name):
            rows = []
            for idx, (img, lbl) in enumerate(zip(images, labels)):
                img_flat = img.flatten().astype(np.float32)
                rows.append((
                    int(idx),
                    int(lbl),
                    label_names_list[int(lbl)],
                    float(img_flat.mean()),
                    float(img_flat.std()),
                    float(img_flat.max()),
                    float(img_flat.min()),
                    split_name
                ))
            return rows

        all_rows = imgs_to_rows(x_train, y_train, "train") + imgs_to_rows(x_test, y_test, "test")

        schema = StructType([
            StructField("image_id",   IntegerType(), False),
            StructField("label",      IntegerType(), False),
            StructField("label_name", StringType(),  False),
            StructField("pixel_mean", FloatType(),   False),
            StructField("pixel_std",  FloatType(),   False),
            StructField("pixel_max",  FloatType(),   False),
            StructField("pixel_min",  FloatType(),   False),
            StructField("split",      StringType(),  False),
        ])
        df = spark.createDataFrame(all_rows, schema=schema)
        print(f"  ✅ Datos cargados desde TensorFlow. Filas: {df.count():,}")
        fuente_datos = "TensorFlow / Keras"
    except Exception as e:
        print(f"  ⚠️  TensorFlow no disponible: {e}")

# --- Intento 3: PyTorch / torchvision ---
if df is None:
    print("Paso 3: Intentando cargar desde PyTorch/torchvision...")
    try:
        import torchvision
        import torch

        label_names_list = ["T-shirt/Top","Trouser","Pullover","Dress","Coat",
                            "Sandal","Shirt","Sneaker","Bag","Ankle Boot"]

        train_ds = torchvision.datasets.FashionMNIST(root="./data", train=True,  download=True)
        test_ds  = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True)

        def torch_ds_to_rows(dataset, split_name):
            rows = []
            for idx in range(len(dataset)):
                img_pil, lbl = dataset[idx]
                arr = np.array(img_pil).flatten().astype(np.float32)
                rows.append((
                    int(idx), int(lbl), label_names_list[int(lbl)],
                    float(arr.mean()), float(arr.std()),
                    float(arr.max()), float(arr.min()),
                    split_name
                ))
            return rows

        all_rows = torch_ds_to_rows(train_ds, "train") + torch_ds_to_rows(test_ds, "test")

        schema = StructType([
            StructField("image_id",   IntegerType(), False),
            StructField("label",      IntegerType(), False),
            StructField("label_name", StringType(),  False),
            StructField("pixel_mean", FloatType(),   False),
            StructField("pixel_std",  FloatType(),   False),
            StructField("pixel_max",  FloatType(),   False),
            StructField("pixel_min",  FloatType(),   False),
            StructField("split",      StringType(),  False),
        ])
        df = spark.createDataFrame(all_rows, schema=schema)
        print(f"  ✅ Datos cargados desde PyTorch. Filas: {df.count():,}")
        fuente_datos = "PyTorch / torchvision"
    except Exception as e:
        print(f"  ⚠️  PyTorch no disponible: {e}")

# --- Intento 4: sklearn fetch_openml ---
if df is None:
    print("Paso 4: Intentando cargar desde sklearn (fetch_openml)...")
    try:
        from sklearn.datasets import fetch_openml
        fmnist = fetch_openml('Fashion-MNIST', version=1, as_frame=True)

        label_names_list = ["T-shirt/Top","Trouser","Pullover","Dress","Coat",
                            "Sandal","Shirt","Sneaker","Bag","Ankle Boot"]

        X_raw = fmnist.data.values.astype(np.float32)
        y_raw = fmnist.target.astype(int).values

        rows = []
        for idx in range(len(X_raw)):
            arr = X_raw[idx]
            lbl = int(y_raw[idx])
            split = "train" if idx < 60000 else "test"
            rows.append((
                idx, lbl, label_names_list[lbl],
                float(arr.mean()), float(arr.std()),
                float(arr.max()), float(arr.min()),
                split
            ))

        schema = StructType([
            StructField("image_id",   IntegerType(), False),
            StructField("label",      IntegerType(), False),
            StructField("label_name", StringType(),  False),
            StructField("pixel_mean", FloatType(),   False),
            StructField("pixel_std",  FloatType(),   False),
            StructField("pixel_max",  FloatType(),   False),
            StructField("pixel_min",  FloatType(),   False),
            StructField("split",      StringType(),  False),
        ])
        df = spark.createDataFrame(rows, schema=schema)
        print(f"  ✅ Datos cargados desde sklearn/OpenML. Filas: {df.count():,}")
        fuente_datos = "sklearn / OpenML"
    except Exception as e:
        print(f"  ⚠️  sklearn/OpenML no disponible: {e}")

# --- Intento 5: Datos sintéticos (siempre disponible) ---
if df is None:
    print("Paso 5: Generando datos sintéticos (réplica estadística de Fashion-MNIST)...")
    df = generar_datos_sinteticos(spark, n=70000)
    fuente_datos = "Datos sintéticos (réplica estadística)"
    print(f"  ✅ Datos sintéticos generados. Filas: {df.count():,}")

# Cachear el DataFrame en memoria para acelerar las operaciones siguientes
df.cache()
df.count()  # forzar materialización del cache

print(f"\n{'='*55}")
print(f"  FUENTE DE DATOS UTILIZADA: {fuente_datos}")
print(f"  Total de registros       : {df.count():,}")
print(f"{'='*55}")

Paso 1: Buscando 'output/leccion4_fashion_data.parquet'...
  ⚠️  Archivo no encontrado en 'output/leccion4_fashion_data.parquet'
Paso 2: Intentando cargar desde TensorFlow/Keras...
  ⚠️  TensorFlow no disponible: An error occurred while calling o47.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (Urzua executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	a

Py4JJavaError: An error occurred while calling o47.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 10 in stage 1.0 failed 1 times, most recent failure: Lost task 10.0 in stage 1.0 (TID 22) (Urzua executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1014)
	at org.apache.spark.api.

---
# Sección 1: Exploración Rápida de los Datos

### ¿Por qué explorar antes de modelar?

Antes de lanzar cualquier modelo, necesitamos responder tres preguntas:
1. **¿Están los datos limpios?** — ¿hay nulos, outliers, tipos incorrectos?
2. **¿Están balanceadas las clases?** — un desbalance severo puede sesgar el clasificador
3. **¿Las features tienen rangos razonables?** — detectar errores de escala o de procesamiento

Este paso ahorra horas de debugging después del entrenamiento.

In [ ]:
# ============================================================
# SECCIÓN 1: Exploración de datos
# ============================================================

print("=" * 60)
print("PASO 1: Primeras 5 filas del dataset")
print("=" * 60)
# show() imprime las primeras N filas en formato tabla
# truncate=False muestra strings completos sin cortar
df.show(5, truncate=False)

In [ ]:
print("=" * 60)
print("PASO 2: Schema del DataFrame")
print("=" * 60)
# printSchema() muestra los tipos de datos de cada columna
# Es importante verificar que pixel_mean, std, etc. sean FloatType
df.printSchema()

In [ ]:
print("=" * 60)
print("PASO 3: Distribución de las 10 clases Fashion-MNIST")
print("=" * 60)
# groupBy + count + orderBy → frecuencia de cada categoría
# Un dataset balanceado tiene ~7,000 imágenes por clase
df.groupBy("label_name", "label") \
  .count() \
  .orderBy("label") \
  .withColumnRenamed("count", "num_imagenes") \
  .show(10, truncate=False)

In [ ]:
print("=" * 60)
print("PASO 4: Estadísticas descriptivas de las 4 features")
print("=" * 60)
# describe() devuelve: count, mean, stddev, min, max
# Miramos si los rangos son consistentes con valores de píxeles [0, 255]
df.describe(["pixel_mean", "pixel_std", "pixel_max", "pixel_min"]).show()

In [ ]:
print("=" * 60)
print("PASO 5: Crear columna 'target' para clasificación binaria")
print("=" * 60)
# Estrategia de negocio RetailMax:
#   CLASE 1 (ropa de cuerpo): labels 0,2,3,4,6
#     → T-shirt/Top, Pullover, Dress, Coat, Shirt
#   CLASE 0 (calzado/accesorios): labels 1,5,7,8,9
#     → Trouser, Sandal, Sneaker, Bag, Ankle Boot
#
# F.when() funciona como IF-THEN-ELSE en SQL
# F.col().isin([...]) es equivalente a SQL's 'IN (...)'

df = df.withColumn(
    "target",
    F.when(F.col("label").isin([0, 2, 3, 4, 6]), 1)  # ropa de cuerpo → 1
     .otherwise(0)                                     # calzado/accesorios → 0
)

# Mostrar distribución del target binario
print("\nDistribución del target binario:")
target_dist = df.groupBy("target").count().orderBy("target")
target_dist.show()

# Añadir descripción textual
print("  target=1 → 'Ropa de cuerpo'   (labels: 0=T-shirt, 2=Pullover, 3=Dress, 4=Coat, 6=Shirt)")
print("  target=0 → 'Calzado/Acces.' (labels: 1=Trouser, 5=Sandal, 7=Sneaker, 8=Bag, 9=Ankle Boot)")

In [ ]:
print("=" * 60)
print("PASO 6: Verificar que no hay valores nulos")
print("=" * 60)
# Contar nulos en cada columna de features + target
# Un nulo en el vector de features rompería el entrenamiento
cols_criticas = ["pixel_mean", "pixel_std", "pixel_max", "pixel_min", "label", "target"]

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(f"nulos_{c}")
    for c in cols_criticas
])
null_counts.show()
print("✅ Si todos los valores son 0, no hay nulos en las columnas críticas")

---
# Sección 2: Ingeniería de Features — VectorAssembler y StandardScaler

## ¿Por qué necesitamos VectorAssembler?

Los algoritmos de MLlib **no trabajan con columnas individuales** como en scikit-learn. Requieren que todas las features estén **agrupadas en una sola columna de tipo `Vector`**. El `VectorAssembler` hace exactamente eso:

```
Antes:  pixel_mean=72.5  pixel_std=45.2  pixel_max=240.0  pixel_min=0.0
Después: features_raw = [72.5, 45.2, 240.0, 0.0]   ← DenseVector
```

## ¿Por qué StandardScaler?

Nuestras features tienen escalas muy distintas:
- `pixel_max` puede llegar a **255**
- `pixel_std` suele estar entre **20 y 80**

Sin estandarizar, `pixel_max` "dominaría" el modelo injustamente. El `StandardScaler` aplica la fórmula **Z-score**:

$$z = \frac{x - \mu}{\sigma}$$

**Mini-ejemplo numérico:**

```
Feature pixel_max:  μ = 200.0,  σ = 50.0
  Imagen A:  pixel_max = 250  →  z = (250 - 200) / 50 =  1.00
  Imagen B:  pixel_max = 150  →  z = (150 - 200) / 50 = -1.00
  Imagen C:  pixel_max = 200  →  z = (200 - 200) / 50 =  0.00  ← promedio

Resultado: todas las features quedan en escala comparable (aprox. -3 a +3)
```

## Pipeline como línea de producción

Imagina una **línea de producción en fábrica**:
```
[Datos brutos] → [VectorAssembler] → [StandardScaler] → [Modelo ML] → [Predicciones]
```

El `Pipeline` de MLlib encadena estos pasos y garantiza que se apliquen en el **mismo orden** tanto en entrenamiento como en producción. Esto evita el clásico error de estandarizar los datos de test con estadísticas del test (en lugar de las del train).

In [ ]:
# ============================================================
# SECCIÓN 2: Ingeniería de Features
# ============================================================

print("=" * 60)
print("PASO 1: Definir las columnas de features")
print("=" * 60)

# Las 4 estadísticas de imagen que calculamos en la Lección 4
feature_cols = ["pixel_mean", "pixel_std", "pixel_max", "pixel_min"]
print(f"Features a usar: {feature_cols}")

print("\nPASO 2: Crear VectorAssembler")
print("-" * 40)
# VectorAssembler combina varias columnas numéricas en un DenseVector
# inputCols: lista de columnas a combinar
# outputCol: nombre de la nueva columna vector
assembler = VectorAssembler(
    inputCols=feature_cols,      # columnas de entrada
    outputCol="features_raw",    # columna de salida (vector sin escalar)
    handleInvalid="skip"         # omitir filas con NaN/Inf en vez de fallar
)
print("  ✅ VectorAssembler creado")
print(f"     Input:  {feature_cols}")
print(f"     Output: 'features_raw' (DenseVector de 4 elementos)")

print("\nPASO 3: Crear StandardScaler")
print("-" * 40)
# StandardScaler aprende μ y σ de los datos de entrenamiento (fit)
# y los aplica a cualquier dataset (transform)
# withMean=True  → centra en 0   (resta μ)
# withStd=True   → escala a σ=1  (divide por σ)
scaler = StandardScaler(
    inputCol="features_raw",  # toma el vector crudo del assembler
    outputCol="features",     # genera el vector estandarizado
    withMean=True,            # restar la media
    withStd=True              # dividir por la desviación estándar
)
print("  ✅ StandardScaler creado")
print(f"     Input:  'features_raw'")
print(f"     Output: 'features' (DenseVector estandarizado)")

print("\nPASO 4: Demostración del vector resultante")
print("-" * 40)
# Construir un mini-pipeline solo para demostrar la transformación
pipeline_demo = Pipeline(stages=[assembler, scaler])
model_demo    = pipeline_demo.fit(df)   # aprende μ y σ de los datos
df_demo       = model_demo.transform(df)  # aplica la transformación

print("  Ejemplo de los vectores de features (3 primeras filas):")
df_demo.select(
    "label_name",
    "pixel_mean", "pixel_std", "pixel_max", "pixel_min",
    "features"
).show(3, truncate=False)

print("\n  Interpretación del vector:")
print("  'features' = [z(pixel_mean), z(pixel_std), z(pixel_max), z(pixel_min)]")
print("  donde z(x) = (x - μ) / σ   →   valores aprox. entre -3 y +3")

---
# Sección 3: Modelo Supervisado — Regresión Logística

## ¿Qué es la Regresión Logística?

A pesar del nombre, la Regresión Logística es un algoritmo de **clasificación**, no de regresión. Estima la **probabilidad** de que una observación pertenezca a la clase positiva (target=1).

### Fórmula:

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w} \cdot \mathbf{x} + b) = \frac{1}{1 + e^{-(w_1x_1 + w_2x_2 + \ldots + w_nx_n + b)}}$$

Donde $\sigma$ es la función **sigmoide** que aplasta cualquier número real al rango (0, 1).

### Mini-ejemplo numérico (con 2 features):

```
Pesos aprendidos: w1=0.8 (pixel_mean), w2=-0.3 (pixel_std), b=-0.5

Imagen X (z_mean=1.2, z_std=0.4):
  logit = 0.8×1.2 + (-0.3)×0.4 + (-0.5)
        = 0.96   - 0.12        - 0.5
        = 0.34

  P(ropa) = 1 / (1 + e^(-0.34)) = 1 / (1 + 0.71) ≈ 0.584

  Como 0.584 > 0.5  →  predicción = 1 (ropa de cuerpo) ✅
```

## Aplicación en RetailMax

RetailMax recibe **miles de nuevos productos** cada semana. El etiquetado manual es costoso. Con este clasificador, el sistema puede:
- **Pre-etiquetar automáticamente** si un producto es "ropa" o "accesorio"
- **Enrutar a la sección correcta** del catálogo sin intervención humana
- **Reducir errores** de catalogación que generan devoluciones

In [ ]:
# ============================================================
# SECCIÓN 3: Regresión Logística con MLlib Pipeline
# ============================================================

print("=" * 60)
print("PASO 1: Dividir datos en Train (80%) y Test (20%)")
print("=" * 60)
# randomSplit asegura que la división sea aleatoria pero reproducible (seed=42)
# IMPORTANTE: siempre usar la misma semilla para comparar experimentos
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Cachear para evitar recalcular en múltiples operaciones
train_df.cache()
test_df.cache()

train_count = train_df.count()
test_count  = test_df.count()
total       = train_count + test_count

print(f"  Train set : {train_count:,} filas  ({100*train_count/total:.1f}%)")
print(f"  Test set  : {test_count:,}  filas  ({100*test_count/total:.1f}%)")
print(f"  Total     : {total:,} filas")

In [ ]:
print("=" * 60)
print("PASO 2: Construir el Pipeline de Clasificación")
print("=" * 60)

# --- Paso 2a: Re-crear assembler y scaler (fresh, sin fit previo) ---
# Es buena práctica crear componentes nuevos para cada pipeline
assembler_lr = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler_lr = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

# --- Paso 2b: Crear el modelo de Regresión Logística ---
# featuresCol: columna que contiene el vector de features
# labelCol   : columna con la etiqueta a predecir (target binario)
# maxIter    : número máximo de iteraciones del optimizador
# regParam   : regularización L2 para evitar overfitting
lr = LogisticRegression(
    featuresCol="features",   # toma el vector estandarizado
    labelCol="target",        # columna creada en Sección 1 (0 o 1)
    predictionCol="prediction",  # columna de predicción (0 o 1)
    probabilityCol="probability",  # columna de probabilidades [P(0), P(1)]
    maxIter=10,               # 10 iteraciones es suficiente para este caso
    regParam=0.01,            # regularización leve
    elasticNetParam=0.0       # 0.0 = solo L2 (Ridge), 1.0 = solo L1 (Lasso)
)

# --- Paso 2c: Encadenar en Pipeline ---
# El Pipeline aplica los stages EN ORDEN:
#   1. assembler_lr → convierte columnas a vector
#   2. scaler_lr    → estandariza el vector
#   3. lr           → entrena el clasificador
pipeline_lr = Pipeline(stages=[assembler_lr, scaler_lr, lr])

print("  Pipeline de Clasificación construido:")
print("    Stage 1: VectorAssembler    → features_raw")
print("    Stage 2: StandardScaler     → features")
print("    Stage 3: LogisticRegression → prediction, probability")

In [ ]:
print("=" * 60)
print("PASO 3: Entrenar el Pipeline")
print("=" * 60)
print("  Entrenando con el 80% de los datos (train_df)...")

# Medir tiempo de entrenamiento
t_inicio = time.time()

# pipeline.fit() hace DOS cosas en cascada:
#   1. Llama a assembler.fit(train_df)  → aprende nada (es un transformador puro)
#   2. Llama a scaler.fit(...)          → aprende μ y σ de train_df
#   3. Llama a lr.fit(...)              → aprende los pesos w del clasificador
# Retorna un PipelineModel (ya entrenado)
model_lr = pipeline_lr.fit(train_df)

t_fin = time.time()
t_entrenamiento = t_fin - t_inicio

print(f"  ✅ Entrenamiento completado")
print(f"     Tiempo: {t_entrenamiento:.2f} segundos")

# Inspeccionar los pesos aprendidos por la Regresión Logística
# model_lr.stages[-1] accede al último stage del Pipeline (el modelo LR)
lr_model = model_lr.stages[-1]
coeficientes = lr_model.coefficients.toArray()
intercepto   = lr_model.intercept

print(f"\n  Pesos aprendidos por el clasificador:")
print(f"  {'Feature':<15} {'Coeficiente':>12}  {'Interpretación':}")
print(f"  {'-'*60}")
interpretaciones = [
    "↑ más brillante → más probable que sea ropa",
    "↑ más variedad tonal → ajusta la predicción",
    "↑ píxel más blanco → ajusta la predicción",
    "↑ píxel más oscuro → ajusta la predicción"
]
for feat, coef, interp in zip(feature_cols, coeficientes, interpretaciones):
    print(f"  {feat:<15} {coef:>12.4f}  {interp}")
print(f"  {'intercepto':<15} {intercepto:>12.4f}")

In [ ]:
print("=" * 60)
print("PASO 4: Predecir sobre el Test Set")
print("=" * 60)
# model_lr.transform() aplica el pipeline completo al test_df:
#   1. assembler transforma las columnas
#   2. scaler aplica la estandarización (con μ,σ del TRAIN, no del test)
#   3. lr genera 'prediction' y 'probability'
predictions_lr = model_lr.transform(test_df)
print("  ✅ Predicciones generadas")

print("\nPASO 5: Muestra de predicciones (5 primeras filas)")
print("-" * 40)
# Mostrar solo las columnas más informativas para el análisis
predictions_lr.select(
    "label_name",     # nombre de la categoría original
    "target",         # etiqueta real (0 o 1)
    "prediction",     # predicción del modelo (0 o 1)
    "probability"     # vector [P(clase=0), P(clase=1)]
).show(5, truncate=False)

print("  Nota: 'probability' es un vector [P(0), P(1)]")
print("        El modelo predice la clase con mayor probabilidad")

---
# Sección 4: Evaluación del Modelo Supervisado

## Métricas de Clasificación

Para evaluar un clasificador binario usamos cuatro métricas principales. Primero definimos la **matriz de confusión**:

```
                    PREDICHO
                  0 (acc.)   1 (ropa)
REAL  0 (acc.)  [  TN   ]  [  FP   ]
      1 (ropa)  [  FN   ]  [  TP   ]

TP = True Positives  (ropa predicha como ropa) ✅
TN = True Negatives  (accesorio predicho como accesorio) ✅
FP = False Positives (accesorio predicho como ropa) ❌
FN = False Negatives (ropa predicha como accesorio) ❌
```

### Fórmulas:

| Métrica | Fórmula | Cuándo importa |
|---------|---------|---------------|
| **Accuracy** | (TP + TN) / Total | Clases balanceadas |
| **Precision** | TP / (TP + FP) | Minimizar falsos positivos |
| **Recall** | TP / (TP + FN) | Minimizar falsos negativos |
| **F1-Score** | 2 × (P × R) / (P + R) | Balance entre precisión y recall |

### ¿Cuál importa más en RetailMax?
- **Precision alta** → evitar etiquetar una zapatilla como "ropa" (mal visual en la tienda)
- **Recall alto** → no perder productos de ropa que deberían aparecer en esa sección
- En catálogos de moda, el **F1-Score** suele ser la métrica de balance más adecuada

In [ ]:
# ============================================================
# SECCIÓN 4: Evaluación del Clasificador
# ============================================================

print("=" * 60)
print("PASO 1: Accuracy con MulticlassClassificationEvaluator")
print("=" * 60)
# MulticlassClassificationEvaluator funciona para binario y multiclase
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="target",          # columna con la etiqueta real
    predictionCol="prediction", # columna con la predicción
    metricName="accuracy"       # métrica a calcular
)
accuracy = evaluator_acc.evaluate(predictions_lr)
print(f"  Accuracy: {accuracy:.4f}  ({accuracy*100:.2f}%)")

# F1-Score también con el evaluador de MLlib
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="f1"
)
f1_score = evaluator_f1.evaluate(predictions_lr)
print(f"  F1-Score: {f1_score:.4f}  ({f1_score*100:.2f}%)")

In [ ]:
print("=" * 60)
print("PASO 2 y 3: Construir la Matriz de Confusión")
print("=" * 60)

# Agrupar por (target real, prediction) y contar frecuencias
# Esto nos da la distribución completa de TP, TN, FP, FN
confusion_spark = predictions_lr.groupBy("target", "prediction").count().orderBy("target", "prediction")
confusion_spark.show()

# Convertir a pandas para manipulación más cómoda
confusion_pd = confusion_spark.toPandas()

# Construir la matriz como pivot table
cm_matrix = confusion_pd.pivot_table(
    index="target",      # filas = clases reales
    columns="prediction", # columnas = predicciones
    values="count",
    fill_value=0
).astype(int)

print("\nMatriz de Confusión (formato pivot):")
print("  Filas = Clase REAL | Columnas = Clase PREDICHA")
print(cm_matrix.to_string())
print("\n  Interpretación:")
print("    (0→0) TN: accesorios/calzado correctamente identificados")
print("    (1→1) TP: ropa de cuerpo correctamente identificada")
print("    (0→1) FP: accesorios/calzado mal clasificados como ropa")
print("    (1→0) FN: ropa de cuerpo mal clasificada como accesorio")

In [ ]:
print("=" * 60)
print("PASO 4: Calcular métricas desde la Matriz de Confusión")
print("=" * 60)

# Extraer TP, TN, FP, FN de la matriz
# Usamos .get() con default 0 para evitar KeyError si alguna celda es 0
try:
    TN = int(cm_matrix.loc[0, 0]) if 0 in cm_matrix.index and 0 in cm_matrix.columns else 0
    FP = int(cm_matrix.loc[0, 1]) if 0 in cm_matrix.index and 1 in cm_matrix.columns else 0
    FN = int(cm_matrix.loc[1, 0]) if 1 in cm_matrix.index and 0 in cm_matrix.columns else 0
    TP = int(cm_matrix.loc[1, 1]) if 1 in cm_matrix.index and 1 in cm_matrix.columns else 0
except Exception:
    # Fallback: calcular desde el DataFrame directamente
    cells = {(int(r["target"]), int(r["prediction"])): int(r["count"])
             for _, r in confusion_pd.iterrows()}
    TN = cells.get((0, 0), 0)
    FP = cells.get((0, 1), 0)
    FN = cells.get((1, 0), 0)
    TP = cells.get((1, 1), 0)

total_test = TP + TN + FP + FN

# Calcular métricas con las fórmulas estándar
# Usamos max(..., 1e-9) para evitar división por cero
precision_manual = TP / max(TP + FP, 1e-9)
recall_manual    = TP / max(TP + FN, 1e-9)
accuracy_manual  = (TP + TN) / max(total_test, 1e-9)
f1_manual        = 2 * precision_manual * recall_manual / max(precision_manual + recall_manual, 1e-9)

print(f"  TP (ropa → ropa correcta)         : {TP:>8,}")
print(f"  TN (accesorio → accesorio correcto): {TN:>8,}")
print(f"  FP (accesorio → ropa [ERROR])      : {FP:>8,}")
print(f"  FN (ropa → accesorio [ERROR])      : {FN:>8,}")
print(f"  Total test                         : {total_test:>8,}")
print()
print(f"  Accuracy  = (TP+TN)/Total = ({TP:,}+{TN:,})/{total_test:,} = {accuracy_manual:.4f}")
print(f"  Precision = TP/(TP+FP)   = {TP:,}/({TP:,}+{FP:,}) = {precision_manual:.4f}")
print(f"  Recall    = TP/(TP+FN)   = {TP:,}/({TP:,}+{FN:,}) = {recall_manual:.4f}")
print(f"  F1-Score  = 2×P×R/(P+R)  = 2×{precision_manual:.4f}×{recall_manual:.4f}/({precision_manual:.4f}+{recall_manual:.4f}) = {f1_manual:.4f}")

In [ ]:
print("=" * 60)
print("PASO 5: Visualización — Heatmap de la Matriz de Confusión")
print("=" * 60)

# Construir la matriz numpy para seaborn
cm_array = np.array([[TN, FP], [FN, TP]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Subplot 1: Counts ---
labels_cm = ["Calzado/Accesorios", "Ropa de Cuerpo"]
sns.heatmap(
    cm_array,
    annot=True,           # mostrar números en las celdas
    fmt=",d",             # formato: entero con separador de miles
    cmap="Blues",         # paleta de azules
    xticklabels=labels_cm,
    yticklabels=labels_cm,
    linewidths=0.5,
    ax=axes[0]
)
axes[0].set_title("Matriz de Confusión\n(Conteos)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicción", fontsize=11)
axes[0].set_ylabel("Valor Real", fontsize=11)

# --- Subplot 2: Porcentajes normalizados por fila ---
cm_norm = cm_array.astype(float) / cm_array.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2%",            # formato porcentaje
    cmap="Greens",
    xticklabels=labels_cm,
    yticklabels=labels_cm,
    linewidths=0.5,
    vmin=0, vmax=1,
    ax=axes[1]
)
axes[1].set_title("Matriz de Confusión\n(Normalizada por fila)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicción", fontsize=11)
axes[1].set_ylabel("Valor Real", fontsize=11)

plt.suptitle("RetailMax — Evaluación del Clasificador Binario\n(Regresión Logística sobre Fashion-MNIST)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()

# Guardar figura
os.makedirs("output", exist_ok=True)
plt.savefig("output/leccion5_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("  ✅ Figura guardada en: output/leccion5_confusion_matrix.png")

In [ ]:
print("=" * 60)
print("PASO 6: Reporte de Métricas — Resumen")
print("=" * 60)

# Crear tabla resumen de métricas con pandas
metricas_df = pd.DataFrame({
    "Métrica":    ["Accuracy", "Precision", "Recall (Sensibilidad)", "F1-Score"],
    "Valor":      [f"{accuracy_manual:.4f}", f"{precision_manual:.4f}",
                   f"{recall_manual:.4f}",   f"{f1_manual:.4f}"],
    "Porcentaje": [f"{accuracy_manual*100:.2f}%", f"{precision_manual*100:.2f}%",
                   f"{recall_manual*100:.2f}%",   f"{f1_manual*100:.2f}%"],
    "Interpetación": [
        f"El {accuracy_manual*100:.1f}% de las predicciones son correctas",
        f"De cada 100 productos marcados 'ropa', {precision_manual*100:.1f} realmente lo son",
        f"El modelo detecta el {recall_manual*100:.1f}% de todos los productos de ropa",
        f"Balance entre precisión y recall: {f1_manual*100:.2f}%"
    ]
})

print("\n" + metricas_df.to_string(index=False))

---
# Sección 5: Modelo No Supervisado — K-Means

## ¿Qué es K-Means?

K-Means es un algoritmo de **clustering (agrupación)** que segmenta datos en **K grupos** sin usar etiquetas. El algoritmo aprende solo desde los patrones en los datos.

### Algoritmo en 4 pasos:

```
1. INICIALIZAR: elegir K puntos aleatorios como centroides iniciales
2. ASIGNAR: cada punto se asigna al centroide más cercano (distancia euclidiana)
3. RECALCULAR: mover cada centroide al promedio de sus puntos asignados
4. REPETIR pasos 2-3 hasta que los centroides no se muevan (convergencia)
```

### Función objetivo (Inercia):

$$\text{Inercia} = \sum_{i=1}^{n} \min_{k} \|\mathbf{x}_i - \mathbf{c}_k\|^2$$

K-Means minimiza esta suma de distancias cuadradas al centroide más cercano.

### Mini-ejemplo numérico (K=2, 6 puntos en 1D):

```
Datos: [2, 3, 4, 8, 9, 10]  →  intuitivamente se forman 2 grupos

Iteración 1:
  Centroides iniciales: c1=2, c2=8
  Asignación: {2,3,4} → c1;  {8,9,10} → c2
  Nuevos centroides: c1=(2+3+4)/3=3.0;  c2=(8+9+10)/3=9.0

Iteración 2:
  Asignación: {2,3,4} → c1=3;  {8,9,10} → c2=9
  Nuevos centroides: c1=3.0;  c2=9.0  ← SIN CAMBIO → CONVERGIÓ ✅

Inercia final: (2-3)²+(3-3)²+(4-3)²+(8-9)²+(9-9)²+(10-9)² = 1+0+1+1+0+1 = 4
```

## Aplicación en RetailMax

K-Means permite a RetailMax:
- **Descubrir segmentos de mercado** naturales en su catálogo visual
- **Crear colecciones temáticas** basadas en similitud visual (no solo categoría)
- **Personalizar recomendaciones**: "clientes que vieron el Cluster A también compraron..."
- **Pricing dinámico**: productos en el mismo cluster tienen valor percibido similar

In [ ]:
# ============================================================
# SECCIÓN 5: K-Means Clustering
# ============================================================

print("=" * 60)
print("PASO 1: Preparar componentes del Pipeline de Clustering")
print("=" * 60)

# Nuevas instancias de assembler y scaler para el pipeline de clustering
# (buena práctica: no reutilizar objetos ya fiteados de otro pipeline)
assembler_km = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler_km = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

print("PASO 2: Configurar K-Means con k=3")
print("-" * 40)
# k=3: buscamos 3 segmentos de mercado en el catálogo visual de RetailMax
# Justificación del k=3:
#   Segmento 1: prendas oscuras/simples (fondo claro, objeto oscuro)
#   Segmento 2: prendas medias/variadas
#   Segmento 3: accesorios/calzado (alta varianza visual)
kmeans = KMeans(
    k=3,                      # número de clusters
    featuresCol="features",   # columna del vector estandarizado
    predictionCol="cluster",  # nombre de la columna de cluster asignado
    seed=42,                  # reproducibilidad
    maxIter=20,               # máximo de iteraciones
    initMode="k-means||",     # inicialización mejorada (k-means++ distribuido)
    distanceMeasure="euclidean"  # distancia euclidiana
)

print("  K-Means configurado:")
print(f"    k            = 3 clusters")
print(f"    initMode     = k-means|| (versión distribuida de k-means++)")
print(f"    maxIter      = 20 iteraciones")
print(f"    seed         = 42")

print("\nPASO 3: Construir Pipeline de Clustering")
print("-" * 40)
# Pipeline: assembler → scaler → kmeans
pipeline_km = Pipeline(stages=[assembler_km, scaler_km, kmeans])
print("  Pipeline construido:")
print("    Stage 1: VectorAssembler  → features_raw")
print("    Stage 2: StandardScaler   → features")
print("    Stage 3: KMeans           → cluster")

In [ ]:
print("=" * 60)
print("PASO 4: Entrenar K-Means sobre todo el dataset")
print("=" * 60)
# K-Means es no supervisado: usa TODOS los datos (sin split train/test)
# No hay 'etiqueta correcta' → evaluamos con métricas de cohesión

t_inicio_km = time.time()
model_km = pipeline_km.fit(df)  # aprende los centroides de los 70,000 productos
t_fin_km = time.time()

print(f"  ✅ K-Means entrenado")
print(f"     Tiempo: {t_fin_km - t_inicio_km:.2f} segundos")

print("\nPASO 5: Asignar clusters a todos los productos")
print("-" * 40)
# transform() asigna a cada imagen su cluster más cercano
clustered_df = model_km.transform(df)
print("  ✅ Clusters asignados")

# Cachear para múltiples operaciones
clustered_df.cache()
clustered_df.count()  # forzar materialización

# Muestra de las asignaciones
print("\n  Primeras 5 filas con cluster asignado:")
clustered_df.select("label_name", "pixel_mean", "pixel_std", "cluster").show(5)

In [ ]:
print("=" * 60)
print("PASO 6: Analizar los centroides aprendidos")
print("=" * 60)

# Acceder al modelo K-Means (último stage del Pipeline)
kmeans_model = model_km.stages[-1]

# clusterCenters() retorna los centroides en el espacio ESTANDARIZADO
# Para interpretarlos, necesitamos revertir la estandarización
centers_scaled = kmeans_model.clusterCenters()

# Obtener μ y σ del StandardScaler para revertir la transformación
scaler_model_km = model_km.stages[1]  # stage 1 = scaler fiteado
mean_vec = scaler_model_km.mean.toArray()
std_vec  = scaler_model_km.std.toArray()

# Revertir estandarización: x_original = z * σ + μ
centers_original = [
    center * std_vec + mean_vec
    for center in centers_scaled
]

print("  Centroides en espacio original (escala de píxeles [0-255]):")
print()
centroides_df = pd.DataFrame(
    centers_original,
    columns=["pixel_mean", "pixel_std", "pixel_max", "pixel_min"]
)
centroides_df.index.name = "Cluster"
centroides_df = centroides_df.round(2)

# Añadir descripción cualitativa basada en pixel_mean
def describir_cluster(mean_val):
    if mean_val < 60:
        return "Productos con fondo oscuro / colores apagados"
    elif mean_val < 90:
        return "Productos de tonos medios / variados"
    else:
        return "Productos brillantes / colores claros"

centroides_df["descripción"] = centroides_df["pixel_mean"].apply(describir_cluster)
print(centroides_df.to_string())
print()
print("  Nota: Los centroides representan el 'producto promedio' de cada segmento")

---
# Sección 6: Evaluación de K-Means + Método del Codo

## ¿Cómo evaluamos un modelo no supervisado?

A diferencia de la clasificación, no tenemos "respuestas correctas" para comparar. Usamos métricas internas:

### Silhouette Score
Mide qué tan bien separado está cada punto de los otros clusters:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

- $a(i)$ = distancia media al propio cluster (cohesión)
- $b(i)$ = distancia media al cluster vecino más cercano (separación)
- **Rango**: -1 (mal asignado) a +1 (perfectamente asignado)
- **Objetivo**: >0.5 es bueno, >0.7 es excelente

### Inercia (WCSS)

$$\text{WCSS} = \sum_{i} \|\mathbf{x}_i - \mathbf{c}_{k(i)}\|^2$$

Suma de distancias cuadradas al centroide asignado. **Menor es mejor**, pero disminuye monotónicamente con k (siempre baja al agregar clusters).

### Método del Codo
Graficamos la inercia vs k y buscamos el punto donde la mejora se vuelve marginal (el "codo" de la curva). Ese k óptimo balancea calidad y complejidad del modelo.

In [ ]:
# ============================================================
# SECCIÓN 6: Evaluación K-Means + Método del Codo
# ============================================================

print("=" * 60)
print("PASO 1: Silhouette Score del modelo k=3")
print("=" * 60)

# ClusteringEvaluator calcula el Silhouette Score usando distancia euclidiana
sil_evaluator = ClusteringEvaluator(
    featuresCol="features",    # vector estandarizado
    predictionCol="cluster",   # columna de cluster asignado
    metricName="silhouette",   # métrica silhouette
    distanceMeasure="squaredEuclidean"  # distancia cuadrática euclidiana
)

silhouette = sil_evaluator.evaluate(clustered_df)
print(f"  Silhouette Score (k=3): {silhouette:.4f}")
print(f"  Interpretación:")
if silhouette > 0.7:
    interp = "Excelente separación entre clusters"
elif silhouette > 0.5:
    interp = "Buena separación entre clusters"
elif silhouette > 0.25:
    interp = "Separación moderada (clusters con algo de solapamiento)"
else:
    interp = "Clusters muy solapados (considerar más features o diferente k)"
print(f"  → {interp}")

print("\nPASO 2: Inercia (WCSS) del modelo k=3")
print("-" * 40)
# trainingCost = suma de distancias cuadradas al centroide (inercia)
inercia_k3 = kmeans_model.summary.trainingCost
print(f"  Inercia (k=3): {inercia_k3:,.2f}")
print(f"  (Menor inercia = puntos más cercanos a sus centroides)")

In [ ]:
print("=" * 60)
print("PASO 3: Método del Codo — probar k de 2 a 7")
print("=" * 60)
print("  Entrenando K-Means para k=2,3,4,5,6,7 ...")
print("  (Esto puede tardar 1-3 minutos)")

k_values  = list(range(2, 8))  # k = 2, 3, 4, 5, 6, 7
inertias  = []                 # lista para guardar inercia por k
sil_scores = []                # lista para silhouette por k

# Preparar el dataset una sola vez (assembler + scaler)
# para no repetir ese trabajo en cada k
pipeline_prep = Pipeline(stages=[assembler_km, scaler_km])

# Necesitamos instancias frescas del assembler y scaler
assembler_elbow = VectorAssembler(
    inputCols=feature_cols, outputCol="features_raw", handleInvalid="skip"
)
scaler_elbow = StandardScaler(
    inputCol="features_raw", outputCol="features", withMean=True, withStd=True
)
prep_model = Pipeline(stages=[assembler_elbow, scaler_elbow]).fit(df)
df_prep    = prep_model.transform(df)  # datos con columna 'features' lista
df_prep.cache()
df_prep.count()

for k in k_values:
    t_k = time.time()
    # Entrenar KMeans directamente sobre df_prep (ya tiene la columna 'features')
    km_k = KMeans(
        k=k, featuresCol="features", predictionCol="cluster",
        seed=42, maxIter=20
    )
    model_k      = km_k.fit(df_prep)
    clustered_k  = model_k.transform(df_prep)

    # Calcular inercia
    inercia_k = model_k.summary.trainingCost
    inertias.append(inercia_k)

    # Calcular silhouette (solo si k > 1)
    try:
        sil_k = ClusteringEvaluator(
            featuresCol="features", predictionCol="cluster",
            metricName="silhouette"
        ).evaluate(clustered_k)
        sil_scores.append(sil_k)
    except Exception:
        sil_scores.append(0.0)

    t_k_elapsed = time.time() - t_k
    print(f"  k={k}: inercia={inercia_k:>12,.1f}  silhouette={sil_scores[-1]:.4f}  ({t_k_elapsed:.1f}s)")

df_prep.unpersist()  # liberar cache
print("  ✅ Evaluación completada")

In [ ]:
print("=" * 60)
print("VISUALIZACIONES: Método del Codo + Silhouette + Clusters")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("RetailMax — Análisis K-Means: Segmentación del Catálogo Visual",
             fontsize=14, fontweight="bold")

# ----- Gráfico 1: Método del Codo (Inercia) -----
ax1 = axes[0, 0]
ax1.plot(k_values, inertias, 'bo-', linewidth=2.5, markersize=8, label="Inercia")
ax1.fill_between(k_values, inertias, alpha=0.1, color='blue')
# Marcar k=3 como el codo elegido
ax1.axvline(x=3, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label="k=3 elegido")
ax1.scatter([3], [inertias[k_values.index(3)]], color='red', s=120, zorder=5)
ax1.set_xlabel("Número de Clusters (k)", fontsize=11)
ax1.set_ylabel("Inercia (WCSS)", fontsize=11)
ax1.set_title("Método del Codo\n(Menor inercia = mejor, pero buscar el 'codo')", fontsize=11)
ax1.set_xticks(k_values)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ----- Gráfico 2: Silhouette Score por k -----
ax2 = axes[0, 1]
colores_sil = ['green' if s == max(sil_scores) else 'steelblue' for s in sil_scores]
bars = ax2.bar(k_values, sil_scores, color=colores_sil, alpha=0.8, edgecolor='white')
ax2.axhline(y=sil_scores[k_values.index(3)], color='red', linestyle='--',
            linewidth=1.5, alpha=0.7, label="k=3")
# Añadir valores encima de las barras
for bar, sil in zip(bars, sil_scores):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
             f'{sil:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_xlabel("Número de Clusters (k)", fontsize=11)
ax2.set_ylabel("Silhouette Score", fontsize=11)
ax2.set_title("Silhouette Score por k\n(Mayor = mejor separación)", fontsize=11)
ax2.set_xticks(k_values)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# ----- Gráfico 3: Scatter pixel_mean vs pixel_std coloreado por cluster -----
ax3 = axes[1, 0]
# Muestrear 2000 puntos para el scatter (toPandas() es costoso con 70k filas)
sample_pd = clustered_df.select("pixel_mean", "pixel_std", "cluster", "label_name") \
                         .sample(fraction=min(2000/70000, 1.0), seed=42) \
                         .toPandas()

colores_cluster = {0: '#e74c3c', 1: '#2ecc71', 2: '#3498db'}
nombres_cluster = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

for cluster_id in [0, 1, 2]:
    mask = sample_pd["cluster"] == cluster_id
    ax3.scatter(
        sample_pd.loc[mask, "pixel_mean"],
        sample_pd.loc[mask, "pixel_std"],
        c=colores_cluster[cluster_id],
        label=nombres_cluster[cluster_id],
        alpha=0.5,
        s=15
    )

# Superponer centroides (en escala original)
for i, center in enumerate(centers_original):
    ax3.scatter(center[0], center[1],
                c=colores_cluster[i], s=300, marker='*',
                edgecolors='black', linewidth=1.5,
                zorder=10, label=f'Centroide {i}')
    ax3.annotate(f'C{i}', (center[0], center[1]),
                 fontsize=12, fontweight='bold', ha='center', va='bottom')

ax3.set_xlabel("pixel_mean (brillo promedio)", fontsize=11)
ax3.set_ylabel("pixel_std (varianza tonal)", fontsize=11)
ax3.set_title("Clusters en el espacio pixel_mean vs pixel_std\n(★ = centroides)", fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# ----- Gráfico 4: Distribución de categorías por cluster -----
ax4 = axes[1, 1]
cluster_label_pd = clustered_df.groupBy("cluster", "label_name").count() \
                                .toPandas() \
                                .sort_values(["cluster", "count"], ascending=[True, False])

# Pivot para barchart apilada
pivot_cl = cluster_label_pd.pivot_table(
    index="cluster", columns="label_name", values="count", fill_value=0
)
# Normalizar por fila para mostrar proporciones
pivot_cl_norm = pivot_cl.div(pivot_cl.sum(axis=1), axis=0)

pivot_cl_norm.plot(
    kind="bar", stacked=True, ax=ax4,
    colormap="tab10", alpha=0.85, edgecolor='white'
)
ax4.set_xlabel("Cluster", fontsize=11)
ax4.set_ylabel("Proporción de imágenes", fontsize=11)
ax4.set_title("Composición de cada cluster\n(proporciones por categoría Fashion-MNIST)", fontsize=11)
ax4.set_xticklabels([f'Cluster {i}' for i in pivot_cl_norm.index], rotation=0)
ax4.legend(fontsize=7, loc='center left', bbox_to_anchor=(1, 0.5))
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("output/leccion5_kmeans_analisis.png", dpi=150, bbox_inches="tight")
plt.show()
print("  ✅ Figura guardada en: output/leccion5_kmeans_analisis.png")

In [ ]:
print("=" * 60)
print("PASO 4: Distribución detallada por cluster")
print("=" * 60)
# Mostrar qué categorías de Fashion-MNIST dominan cada cluster
print("  Top 3 categorías por cluster:")
print()

for cluster_id in sorted(cluster_label_pd["cluster"].unique()):
    top3 = (
        cluster_label_pd[cluster_label_pd["cluster"] == cluster_id]
        .sort_values("count", ascending=False)
        .head(3)
    )
    total_cluster = top3["count"].sum()  # aprox, solo top3
    print(f"  Cluster {cluster_id} (centroide pixel_mean ≈ {centers_original[cluster_id][0]:.1f}):")
    for _, row in top3.iterrows():
        print(f"    - {row['label_name']:<20} : {row['count']:,} imágenes")
    print()

---
# Sección 7: Insights para Marketing de RetailMax

## Traducir Resultados Técnicos a Lenguaje de Negocio

Los datos solos no crean valor — su interpretación estratégica sí. En esta sección traducimos los números del modelo a recomendaciones concretas para el equipo de marketing de RetailMax.

In [ ]:
# ============================================================
# SECCIÓN 7: Reporte de Marketing para RetailMax
# ============================================================

print("\n" + "="*70)
print("        REPORTE EJECUTIVO PARA MARKETING — RETAILMAX")
print("        Análisis de Catálogo Visual con Machine Learning")
print("="*70)

# ----- Tabla 1: Clasificador Binario -----
print("\n📊 TABLA 1: CLASIFICACIÓN AUTOMÁTICA DEL CATÁLOGO")
print("-" * 50)

# Calcular distribución de categorías en el catálogo completo
target_counts = df.groupBy("target").count().toPandas()
total_cat = target_counts["count"].sum()

ropa_count = int(target_counts[target_counts["target"] == 1]["count"].values[0])
acc_count  = int(target_counts[target_counts["target"] == 0]["count"].values[0])

clasificador_df = pd.DataFrame({
    "Segmento": ["Ropa de Cuerpo", "Calzado y Accesorios"],
    "Categorías incluidas": [
        "T-shirt, Pullover, Dress, Coat, Shirt",
        "Trouser, Sandal, Sneaker, Bag, Ankle Boot"
    ],
    "Productos en catálogo": [f"{ropa_count:,}", f"{acc_count:,}"],
    "Porcentaje": [f"{100*ropa_count/total_cat:.1f}%", f"{100*acc_count/total_cat:.1f}%"],
    "Precisión clasificador": [f"{precision_manual*100:.1f}%", f"N/A (clase negativa)"],
})
print(clasificador_df.to_string(index=False))

print(f"\n  → Accuracy global del clasificador: {accuracy_manual*100:.2f}%")
print(f"  → De cada 100 productos nuevos, el sistema etiqueta correctamente")
print(f"    ~{accuracy_manual*100:.0f} de ellos sin intervención humana.")

# ----- Tabla 2: Segmentos K-Means -----
print("\n\n📊 TABLA 2: SEGMENTOS DE MERCADO IDENTIFICADOS (K-Means, k=3)")
print("-" * 50)

# Construir nombres descriptivos basados en pixel_mean de cada centroide
def nombre_segmento(centroide):
    pm = centroide[0]  # pixel_mean
    ps = centroide[1]  # pixel_std
    if pm < 60:
        return "Prendas Oscuras & Accesorios Complejos"
    elif pm < 85:
        return "Prendas de Tono Medio & Calzado"
    else:
        return "Prendas Claras & Colecciones Premium"

def recomendacion_marketing(centroide):
    pm = centroide[0]
    if pm < 60:
        return ("Campaña 'Noche y Estilo': fondos oscuros, \n"
                "énfasis en contrastes. Ideal para \n"
                "Instagram ads en modo oscuro.")
    elif pm < 85:
        return ("Campaña 'Everyday Essentials': \n"
                "productos versátiles. Newsletter mensual \n"
                "con bundles de outfit completo.")
    else:
        return ("Campaña 'Bright Collection': \n"
                "colores vibrantes. Ideal para \n"
                "primavera/verano. Push en Pinterest.")

# Contar productos por cluster
cluster_counts = clustered_df.groupBy("cluster").count().toPandas()
cluster_counts = cluster_counts.sort_values("cluster")

print()
for _, row in cluster_counts.iterrows():
    cid   = int(row["cluster"])
    cnt   = int(row["count"])
    pct   = 100 * cnt / total_cat
    c     = centers_original[cid]
    nombre = nombre_segmento(c)
    rec    = recomendacion_marketing(c)

    print(f"  ╔═ SEGMENTO {cid}: {nombre}")
    print(f"  ║  Tamaño        : {cnt:,} productos ({pct:.1f}% del catálogo)")
    print(f"  ║  pixel_mean    : {c[0]:.1f}  ({'oscuro' if c[0]<70 else 'medio' if c[0]<90 else 'claro'})")
    print(f"  ║  pixel_std     : {c[1]:.1f}  ({'alta varianza' if c[1]>50 else 'varianza moderada'})")
    print(f"  ║  Recomendación : {rec}")
    print(f"  ╚{'═'*60}")
    print()

# ----- Métrica 3: Resumen de valor de negocio -----
print("\n📊 TABLA 3: VALOR DE NEGOCIO DEL PIPELINE ML")
print("-" * 50)

# Estimación de ahorro basada en porcentaje de automatización
tiempo_manual_min = 2  # minutos por producto (etiquetado manual estimado)
productos_nuevos_mes = 5000  # estimación productos nuevos/mes en RetailMax
pct_automatizable = accuracy_manual
productos_automatizados = int(productos_nuevos_mes * pct_automatizable)
horas_ahorradas = (productos_automatizados * tiempo_manual_min) / 60

valor_df = pd.DataFrame({
    "Indicador": [
        "Precisión del clasificador automático",
        "Productos etiquetables automáticamente/mes",
        "Horas de trabajo manual ahorradas/mes",
        "Silhouette Score de segmentación (k=3)",
        "Segmentos de mercado identificados",
    ],
    "Valor": [
        f"{accuracy_manual*100:.1f}%",
        f"{productos_automatizados:,} de {productos_nuevos_mes:,}",
        f"{horas_ahorradas:.0f} horas",
        f"{silhouette:.4f} (>0.3 = aceptable)",
        "3 segmentos con estrategias diferenciadas",
    ]
})
print(valor_df.to_string(index=False))

print("\n" + "="*70)
print("FIN DEL REPORTE EJECUTIVO")
print("="*70)

---
# Sección 8: Guardar Modelos y Resultados

Guardamos los resultados del pipeline para uso futuro: predicciones del clasificador y asignación de clusters. Estos archivos son los entregables finales del proyecto RetailMax.

In [ ]:
# ============================================================
# SECCIÓN 8: Guardar Resultados en Parquet
# ============================================================

os.makedirs("output", exist_ok=True)  # crear carpeta si no existe

# ----- PASO 1: Guardar predicciones del clasificador LR -----
print("PASO 1: Guardando predicciones del clasificador (Regresión Logística)...")
PATH_LR = "output/leccion5_predicciones_lr.parquet"
try:
    # Seleccionar columnas relevantes para guardar (no el vector de features)
    cols_guardar_lr = ["image_id", "label", "label_name", "target",
                       "prediction", "pixel_mean", "pixel_std", "pixel_max", "pixel_min"]
    predictions_lr_save = predictions_lr.select(cols_guardar_lr)

    # overwrite: sobreescribir si el archivo ya existe
    predictions_lr_save.write.mode("overwrite").parquet(PATH_LR)
    n_saved_lr = predictions_lr_save.count()
    print(f"  ✅ Guardado: {PATH_LR}")
    print(f"     Filas guardadas: {n_saved_lr:,}")
except Exception as e:
    print(f"  ❌ Error guardando predicciones LR: {e}")

# ----- PASO 2: Guardar segmentación K-Means -----
print("\nPASO 2: Guardando segmentación K-Means...")
PATH_KM = "output/leccion5_clusters_km.parquet"
try:
    # Guardar solo columnas relevantes (sin los vectores intermedios)
    cols_guardar_km = ["image_id", "label", "label_name", "cluster",
                       "pixel_mean", "pixel_std", "pixel_max", "pixel_min"]
    clustered_save = clustered_df.select(cols_guardar_km)

    clustered_save.write.mode("overwrite").parquet(PATH_KM)
    n_saved_km = clustered_save.count()
    print(f"  ✅ Guardado: {PATH_KM}")
    print(f"     Filas guardadas: {n_saved_km:,}")
except Exception as e:
    print(f"  ❌ Error guardando clusters KM: {e}")

# ----- PASO 3: Inventario de archivos del proyecto -----
print("\nPASO 3: Inventario de archivos generados en output/")
print("-" * 50)

if os.path.exists("output"):
    archivos = []
    for root, dirs, files in os.walk("output"):
        for f in sorted(files):
            ruta = os.path.join(root, f)
            try:
                size_bytes = os.path.getsize(ruta)
                if size_bytes >= 1024 * 1024:
                    size_str = f"{size_bytes / (1024*1024):.1f} MB"
                elif size_bytes >= 1024:
                    size_str = f"{size_bytes / 1024:.1f} KB"
                else:
                    size_str = f"{size_bytes} B"
                archivos.append((ruta, size_str))
            except Exception:
                archivos.append((ruta, "?"))

    # Agrupar por carpeta/archivo principal
    inventario_df = pd.DataFrame(archivos, columns=["Archivo", "Tamaño"])

    print("\n" + inventario_df.to_string(index=False))
    print(f"\n  Total de archivos: {len(archivos)}")
else:
    print("  ⚠️  Carpeta 'output/' no encontrada")

print("\n✅ Todos los resultados del pipeline han sido guardados")

---
# Sección 9: Cerrar Spark

Es importante cerrar la SparkSession al finalizar para liberar los recursos del cluster (memoria, threads, conexiones de red). En entornos de producción, no cerrar Spark puede causar fugas de memoria.

In [ ]:
# ============================================================
# SECCIÓN 9: Cerrar SparkSession
# ============================================================

# Liberar caches antes de cerrar para evitar mensajes de advertencia
try:
    df.unpersist()             # liberar df principal
    train_df.unpersist()       # liberar train set
    test_df.unpersist()        # liberar test set
    clustered_df.unpersist()   # liberar clustering results
    print("✅ Caches liberados")
except Exception as e:
    print(f"  (Aviso al liberar caches: {e})")

# Cerrar la sesión de Spark
spark.stop()

print("\n✅ SparkSession cerrada correctamente")
print("   Los recursos del driver han sido liberados")
print("\n" + "="*50)
print("   LECCIÓN 5 COMPLETADA — Módulo 9 FINALIZADO")
print("   RetailMax Analytics Pipeline — Proyecto Final")
print("="*50)

---
# Sección 10: Informe Final del Módulo 9

---

## 🏆 Resumen Ejecutivo del Proyecto Completo

A lo largo de las **5 lecciones del Módulo 9**, construiste un pipeline completo de análisis de datos a escala industrial usando **Apache Spark** y **MLlib**. El proyecto simuló el caso real de **RetailMax**, una empresa de e-commerce de moda que necesita procesar y analizar 70,000 imágenes de productos de forma automatizada.

---

## 📋 Recorrido del Proyecto: Lección por Lección

| Lección | Técnica aprendida | Herramienta Spark | Resultado obtenido | Aplicación RetailMax |
|---------|-------------------|-------------------|--------------------|---------------------|
| **L1** | Big Data & arquitectura distribuida | SparkContext, deploy modes | Comprensión de escala e infraestructura | Dimensionar infraestructura para catálogo de millones de productos |
| **L2** | SparkSession y operaciones básicas | SparkSession, DataFrames | Primera transformación distribuida | Primeras consultas sobre el catálogo de moda |
| **L3** | RDDs: transformaciones y acciones | `map`, `filter`, `reduceByKey`, `flatMap` | Pipeline de conteo y agregación a nivel de píxeles | Procesamiento de logs de clics y compras por categoría |
| **L4** | DataFrames, Spark SQL y Parquet | `DataFrame`, `createOrReplaceTempView`, `.parquet()` | `leccion4_fashion_data.parquet`, estadísticas de imagen | Catálogo persistido con estadísticas para reportes BI |
| **L5** | ML Pipeline con MLlib | `Pipeline`, `VectorAssembler`, `LR`, `KMeans` | Clasificador (acc. ≥87%) + 3 segmentos de mercado | Etiquetado automático y segmentación para campañas |

---

## 🔢 Logros Técnicos del Pipeline

- **70,000 imágenes procesadas** con estadísticas de píxeles extraídas y persistidas en Parquet
- **Clasificador binario** (Regresión Logística) con ~87–92% de accuracy en el test set
- **Segmentación en 3 grupos** con K-Means para campañas de marketing diferenciadas
- **Pipeline reproducible**: el mismo código clasifica correctamente productos nuevos sin reentrenar
- **Estimación de automatización**: ~87–92% de los nuevos productos pueden etiquetarse sin revisión manual

---

## 💡 Conceptos Clave Dominados

### Feature Engineering
- `VectorAssembler`: combinar múltiples columnas en un solo vector de features
- `StandardScaler`: estandarización Z-score para igualar escalas
- `StringIndexer`: codificar variables categóricas como índices numéricos

### Modelos Supervisados
- **Regresión Logística**: clasificación binaria con función sigmoide
- Métricas: Accuracy, Precision, Recall, F1-Score, Matriz de Confusión

### Modelos No Supervisados
- **K-Means**: segmentación por minimización de inercia
- Evaluación: Silhouette Score, Método del Codo
- Interpretación de centroides en espacio original

### Infraestructura de ML a Escala
- `Pipeline` de MLlib: encadenar preprocesamiento + modelo en un solo objeto reproducible
- Serialización con Parquet: guardar resultados intermedios y finales
- `CrossValidator` + `ParamGridBuilder`: optimización de hiperparámetros (conocimiento disponible)

---

## 🚀 Próximos Pasos Sugeridos

### Mejoras al Clasificador
1. **Más features**: añadir entropía de la imagen, número de regiones, histograma de colores
2. **Clasificación multiclase**: predecir las 10 categorías en lugar del binario
3. **Random Forest / GBT**: probar `RandomForestClassifier` o `GBTClassifier` de MLlib
4. **Cross-validation completa**: usar `CrossValidator` + `ParamGridBuilder` para tuning

### Mejoras al Clustering
1. **BisectingKMeans**: alternativa jerárquica disponible en MLlib para clusters más definidos
2. **DBSCAN / GMM**: explorar clustering basado en densidad o mezcla de Gaussianas
3. **Features más ricas**: incluir histogramas de 8 bins como features (11 features totales)
4. **Optimal k automático**: automatizar el método del codo con detección de punto de inflexión

### Escala a Producción
1. **Spark Streaming**: procesar nuevas imágenes en tiempo real conforme llegan al catálogo
2. **MLflow**: registrar experimentos, comparar modelos, hacer versionado
3. **Databricks / AWS EMR**: escalar el cluster para catálogos de millones de productos
4. **Serving**: exportar el modelo a ONNX o TensorFlow Serving para predicción en tiempo real

---

## 🗂️ Conexión con el Portafolio — GitHub

Este proyecto está diseñado para destacar en entrevistas técnicas. Para publicarlo:

```
RetailMax-Retail-Analytics-Pipeline/
├── README.md                    ← Descripción del proyecto, screenshots, instrucciones
├── requirements.txt             ← pyspark, numpy, pandas, matplotlib, seaborn
├── notebooks/
│   ├── leccion1_bigdata.ipynb
│   ├── leccion2_spark_intro.ipynb
│   ├── leccion3_rdds.ipynb
│   ├── leccion4_dataframes_sql_parquet.ipynb
│   └── leccion5_mllib_pipeline.ipynb   ← ¡Este notebook!
├── output/                      ← .gitignore o guardar 1 muestra pequeña
│   ├── leccion4_fashion_data.parquet
│   ├── leccion5_predicciones_lr.parquet
│   ├── leccion5_clusters_km.parquet
│   ├── leccion5_confusion_matrix.png
│   └── leccion5_kmeans_analisis.png
└── scripts/
    └── leccion5_mllib_pipeline.py   ← Script equivalente
```

**Tips para el README:**
- Incluir la imagen `leccion5_kmeans_analisis.png` directamente
- Citar el accuracy del clasificador como "logro cuantificable"
- Mencionar: Apache Spark, PySpark, MLlib, Fashion-MNIST, 70K registros
- Agregar badge de Python version y PySpark version

---

## ✅ Checklist Final del Proyecto Completo

### Lección 1 — Big Data
- [ ] Comprendo los conceptos de datos distribuidos y su necesidad en e-commerce
- [ ] Conozco la arquitectura Driver/Executor de Spark
- [ ] Entiendo la diferencia entre Hadoop MapReduce y Spark

### Lección 2 — Apache Spark Intro
- [ ] Puedo crear y configurar una SparkSession
- [ ] Sé la diferencia entre transformaciones lazy y acciones
- [ ] Comprendo el concepto de DAG de ejecución

### Lección 3 — RDDs
- [ ] Domino `map`, `filter`, `flatMap`, `reduceByKey`, `groupByKey`
- [ ] Sé cuándo usar RDDs vs DataFrames
- [ ] Puedo construir un pipeline ETL con RDDs

### Lección 4 — DataFrames, SQL y Parquet
- [ ] Domino las operaciones de DataFrame (select, filter, groupBy, join)
- [ ] Puedo usar Spark SQL con `createOrReplaceTempView`
- [ ] Sé leer y escribir archivos Parquet
- [ ] Guardé `output/leccion4_fashion_data.parquet` ✅

### Lección 5 — MLlib Pipeline (Esta lección)
- [ ] Entiendo el patrón Pipeline de MLlib (Estimator/Transformer)
- [ ] Puedo usar `VectorAssembler` y `StandardScaler` para feature engineering
- [ ] Entrené un clasificador de Regresión Logística y evalué sus métricas
- [ ] Construí un modelo K-Means y analicé los clusters con el método del codo
- [ ] Traduje los resultados técnicos a insights de negocio para marketing
- [ ] Guardé `output/leccion5_predicciones_lr.parquet` ✅
- [ ] Guardé `output/leccion5_clusters_km.parquet` ✅
- [ ] Generé las 4 visualizaciones (confusion matrix, codo, scatter, distribución) ✅

---

> **¡Felicitaciones!** Has completado el **Módulo 9 — Retail Analytics Pipeline**. Ahora posees una base sólida en Big Data con Apache Spark y Machine Learning escalable con MLlib. Estas habilidades son altamente demandadas en el mercado laboral de Data Engineering y Data Science. 🎯